# Exercise 2.6 — Pandas Joins and Merges

This exercise uses the Zambia ECON 2025 establishment census (10% sample) and fake supplementary tables to practice:
- `pd.merge` (left, inner) with different key types
- `pd.concat` for stacking DataFrames
- Join QA: checking for key mismatches, NaN rates, and silent row multiplication

### Setup (run first)

In [ ]:
import os
import pandas as pd
import pyreadstat

DATA_RAW    = '../../data/0_raw/zambia'
DATA_OUT    = '../../data/02_processed'

sav_path = os.path.join(DATA_RAW, 'ECON2025_10percent_120326.sav')
print('Exists?', os.path.exists(sav_path))

df_raw, meta = pyreadstat.read_sav(sav_path, encoding='latin1')

# Keep only the columns we need for this exercise
COLS = ['interview__key', 'G1_prov', 'G2_dist', 'isic_2dgts', 'isic4']
df = df_raw[COLS].copy()

# G1_prov comes in as float — convert to int for cleaner keys
df['G1_prov'] = df['G1_prov'].astype('Int64')   # nullable integer
df['isic_2dgts'] = df['isic_2dgts'].astype('Int64')

print(f'Shape: {df.shape}')
df.head()

---

## Task 1 — Create and merge a province lookup table

Create a small reference table that maps province codes to province names and regions, then merge it into `df`.

> **Hint:** use `how='left'` so every row in `df` is kept even if no match is found.

In [ ]:
prov_lookup = pd.DataFrame({
    'prov_code': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'province_name': [
        'Central', 'Copperbelt', 'Eastern', 'Luapula', 'Lusaka',
        'Muchinga', 'Northern', 'North-Western', 'Southern', 'Western'
    ],
    'region': [
        'Central', 'Copperbelt', 'Eastern', 'Northern', 'Lusaka',
        'Northern', 'Northern', 'Western', 'Southern', 'Western'
    ]
})

# QA before merging
print('Left keys missing :', df['G1_prov'].isna().sum())
print('Left keys unique  :', df['G1_prov'].nunique())
print('Right keys unique :', prov_lookup['prov_code'].nunique())

In [ ]:
rows_before = len(df)

df_merged = # your code here — merge df with prov_lookup on the correct keys

rows_after = len(df_merged)
print('Rows before:', rows_before, '| after:', rows_after)
if rows_after != rows_before:
    print('⚠️  Row count changed by', rows_after - rows_before)

df_merged[['G1_prov', 'province_name', 'region']].head(8)

In [ ]:
# How many rows have a NaN in province_name after the merge?
# your code here

**Questions:**

- Did the row count change after the merge? Why or why not?
- How many rows have `NaN` in `province_name`? What does that tell you about the data?

---

## Task 2 — Create and merge an ISIC description lookup

Create a lookup table for 2-digit ISIC codes and merge it with `df_merged`.

> Note: the lookup covers only a subset of codes — many establishments will not match.

In [ ]:
isic_lookup = pd.DataFrame({
    'isic_2d': [1, 10, 25, 41, 45, 46, 47, 55, 56, 62],
    'isic_description': [
        'Crop production', 'Food products', 'Fabricated metals',
        'Building construction', 'Motor vehicle trade', 'Wholesale trade',
        'Retail trade', 'Accommodation', 'Food & beverage service',
        'Computer programming'
    ]
})

# QA
print('ISIC lookup unique codes:', isic_lookup['isic_2d'].nunique())
print('Main data unique codes  :', df_merged['isic_2dgts'].nunique())

In [ ]:
df_merged = # your code here — merge df_merged with isic_lookup

print(df_merged['isic_description'].value_counts().head(10))

In [ ]:
# Calculate the match rate: what share of rows got a non-NaN description?
match_rate = # your code here
print(f'Match rate: {match_rate:.1%}')
print(f'No match  : {1 - match_rate:.1%}')

**Questions:**

- What share of rows got a match? What share are `NaN`?
- Why is a `left` join the right choice here (vs `inner`)?

---

## Task 3 — Split and concatenate

Simulate receiving data in two batches (Lusaka vs. the rest), then rejoin them.

In [ ]:
df_lusaka = df[df['G1_prov'] == 5].copy()   # province 5 = Lusaka
df_other  = df[df['G1_prov'] != 5].copy()

print(f'Lusaka: {len(df_lusaka)} | Other: {len(df_other)} | Total: {len(df)}')

In [ ]:
# Concatenate the two batches back together
df_combined = # your code here — use pd.concat with ignore_index=True

print(f'Combined: {len(df_combined)} | Original: {len(df)}')
if len(df_combined) != len(df):
    print('⚠️  Row counts do not match!', len(df_combined) - len(df))

**Questions:**

- Why do we use `ignore_index=True` when concatenating?
- What would happen if `df_lusaka` and `df_other` had different column sets?

---

## Task 4 — Join QA: detect duplicates created by a lookup table

A common silent bug: the right-hand lookup table contains duplicate keys, which multiplies rows on merge.

In [ ]:
# A deliberately bad lookup — province 1 appears twice
prov_lookup_bad = pd.DataFrame({
    'prov_code': [1, 1, 2, 3],
    'province_name': ['Central', 'Central (duplicate)', 'Copperbelt', 'Eastern'],
    'region': ['Central', 'Central', 'Copperbelt', 'Eastern'],
})

# Inspect the duplicates in the right table
print('Duplicate keys in bad lookup:', prov_lookup_bad['prov_code'].duplicated().sum())
print(prov_lookup_bad[prov_lookup_bad['prov_code'].duplicated(keep=False)])

In [ ]:
# Merge with the bad lookup — watch the row count!
rows_before = len(df)
df_test = pd.merge(df, prov_lookup_bad, left_on='G1_prov', right_on='prov_code', how='left')
rows_after = len(df_test)

print('Rows before:', rows_before, '| after:', rows_after)
if rows_after != rows_before:
    print('⚠️  Row count changed by', rows_after - rows_before)

In [ ]:
# Fix: deduplicate the lookup table, then merge again
prov_lookup_fixed = # your code here — drop duplicates on 'prov_code', keep first

rows_before = len(df)
df_test_fixed = pd.merge(df, prov_lookup_fixed, left_on='G1_prov', right_on='prov_code', how='left')
rows_after = len(df_test_fixed)

print('Rows before:', rows_before, '| after:', rows_after)

**Questions:**

- Why did the first merge silently change the row count?
- In real projects, should you fix duplicates with `drop_duplicates`, or should you fix the source data? Why?

---

## Task 5 — Build fake trade data and practice concatenation + many-to-one join

Create two DataFrames (imports and exports) with the same schema, concatenate them, then enrich with a product lookup.

In [ ]:
imports = pd.DataFrame({
    'year':        [2024, 2024, 2025, 2025, 2025],
    'partner':     ['ZAF', 'TZA', 'ZAF', 'MOZ', 'TZA'],
    'hs2':         ['10', '10', '10', '12', '12'],
    'value_local': [120000, 80000, 150000, 50000, 40000],
})
imports['flow'] = 'import'

exports = pd.DataFrame({
    'year':        [2024, 2024, 2025, 2025],
    'partner':     ['ZAF', 'COD', 'ZAF', 'COD'],
    'hs2':         ['10', '10', '12', '10'],
    'value_local': [60000, 30000, 70000, 45000],
})
exports['flow'] = 'export'

# Concatenate imports and exports into one DataFrame
trade = # your code here
print(trade)

In [ ]:
hs_lookup = pd.DataFrame({
    'hs2':     ['10', '12'],
    'product': ['Cereals', 'Oil seeds'],
})

# QA: verify no duplicate keys in the lookup
n_dups = hs_lookup['hs2'].duplicated().sum()
print('HS lookup duplicate keys:', n_dups)

# Merge the product description into trade
trade = # your code here — merge trade with hs_lookup on 'hs2'
print(trade)

**Questions:**

- What are the join keys, and why is this a many-to-one merge?
- How would you detect if `hs_lookup` had duplicates and was silently multiplying rows?

---

## Task 6 — Multi-key join: currency conversion by year

Add an exchange rate table and convert trade values to USD. This requires joining on **multiple columns** simultaneously.

In [ ]:
fx = pd.DataFrame({
    'year':          [2024, 2024, 2025, 2025],
    'currency':      ['MWK', 'ZMW', 'MWK', 'ZMW'],
    'usd_per_local': [0.00058, 0.041, 0.00055, 0.039],
})

# Add currency column to trade
trade['currency'] = 'ZMW'

# QA: check the fx table has no duplicate (year, currency) combinations
print('FX duplicate (year, currency):', fx.duplicated(['year', 'currency']).sum())

In [ ]:
# Merge trade with fx on BOTH year and currency
trade_fx = # your code here

# Check: every row should find an FX rate
print('Rows missing FX rate:', trade_fx['usd_per_local'].isna().sum())

trade_fx['value_usd'] = trade_fx['value_local'] * trade_fx['usd_per_local']

# Row count should be unchanged
print(f'trade: {len(trade)} | trade_fx: {len(trade_fx)}')

trade_fx

**Questions:**

- Why is `on=['year', 'currency']` required here, rather than joining on just one column?
- What would happen if `fx` had two rows for `(2025, ZMW)`? How would you detect and prevent it?

---

## Task 7 — Export the enriched dataset

Save the fully enriched `df_merged` to the processed data folder.

In [ ]:
df_final = df_merged.reset_index(drop=True)

out_path = os.path.join(DATA_OUT, 'econ2025_analysis_ready.csv')
df_final.to_csv(out_path, index=False)
print(f'Saved: {out_path} ({df_final.shape[0]} rows, {df_final.shape[1]} cols)')

**Document:** In the cell below, write a short summary of what was merged and why.

_Your summary here._